# AML-GraphMiner — Entraînement GCN / GraphSAGE / GAT sur **IBM AML HI-Small**

Ce notebook :
1. charge le dataset IBM AML **HI-Small** et fait un split temporel 60/20/20 ;
2. construit les features d'arêtes (transactions) et de nœuds (comptes) ;
3. entraîne les **3 architectures** (GCN, GraphSAGE, GAT) pour la classification de transactions ;
4. évalue chaque modèle (PR-AUC, F1, précision, rappel, recall@k) et sauvegarde poids + config ;
5. **Partie inférence** : recharge les modèles et score des transactions choisies par l'utilisateur.

> Le notebook jumeau fait exactement la même chose sur **LI-Small**.

Runtime conseillé : **GPU (T4)** dans Colab.

## 1. Installation

In [ ]:
import torch
TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
print(TORCH, CUDA)

!pip -q install torch_geometric
!pip -q install pyg_lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip -q install kagglehub scikit-learn

## 2. Configuration

In [ ]:
import os

DATASET = "HI-Small"
MODELS  = ["gcn", "sage", "gat"]   # les 3 architectures demandées
SEEDS   = [0]                      # mettez [0, 1, 2] pour moyenne ± écart-type
EPOCHS  = 10
LR      = 1e-3
POS_W   = 30.0                     # poids de la classe illicite (déséquilibre ~0.1 %)
HID, LAYERS, DROP = 64, 2, 0.2
NUM_NEIGHBORS = [10, 5]
BATCH_SIZE = 4096

# Dossier de sortie (poids, configs, résultats)
from google.colab import drive
drive.mount('/content/drive')
FOLDER = "/content/drive/MyDrive"
OUT = os.path.join(FOLDER, "delivery", DATASET)
os.makedirs(OUT, exist_ok=True)

CSV = os.path.join(FOLDER, f"{DATASET}_Trans.csv")
# Alternative : téléchargement direct depuis Kaggle
# import kagglehub
# path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
# CSV = os.path.join(path, f"{DATASET}_Trans.csv")

assert os.path.exists(CSV), f"Fichier introuvable : {CSV}"
print(CSV, round(os.path.getsize(CSV) / 1e6, 1), "Mo")

## 3. Fonctions communes (features + modèle)

`build_features` sert **à la fois** à l'entraînement (statistiques calculées sur le train)
et à l'inférence (statistiques rechargées depuis la config sauvegardée), pour garantir
que les données utilisateur sont transformées exactement comme les données d'entraînement.

In [ ]:
import json, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from sklearn.metrics import average_precision_score, precision_recall_curve

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device :", device)

IBM_COLUMNS = ["Timestamp", "From Bank", "Account", "To Bank", "Account.1", "Amount Received",
               "Receiving Currency", "Amount Paid", "Payment Currency", "Payment Format", "Is Laundering"]
SHORT = ["ts", "from_bank", "from_acc", "to_bank", "to_acc", "amt_recv",
         "cur_recv", "amt_paid", "cur_paid", "fmt", "y"]

def standardize_columns(df):
    """Renomme les colonnes IBM AML en noms courts ; 'Is Laundering' est optionnelle."""
    df = df.copy()
    if len(df.columns) == 10:
        df["y"] = np.nan
    df.columns = SHORT
    df["ts"] = pd.to_datetime(df["ts"])
    df["src_key"] = df["from_bank"].astype(str) + "_" + df["from_acc"].astype(str)
    df["dst_key"] = df["to_bank"].astype(str) + "_" + df["to_acc"].astype(str)
    return df

def index_accounts(df):
    """Un identifiant entier par compte (banque + numéro)."""
    codes, uniques = pd.factorize(pd.concat([df["src_key"], df["dst_key"]]))
    E = len(df)
    return codes[:E], codes[E:], len(uniques), uniques

def _raw_edge_features(df):
    hour = df["ts"].dt.hour + df["ts"].dt.minute / 60
    ef = pd.DataFrame({
        "log_paid": np.log1p(df["amt_paid"].values),
        "log_recv": np.log1p(df["amt_recv"].values),
        "same_cur": (df["cur_paid"] == df["cur_recv"]).astype(float).values,
        "same_bank": (df["from_bank"] == df["to_bank"]).astype(float).values,
        "h_sin": np.sin(2 * np.pi * hour / 24).values,
        "h_cos": np.cos(2 * np.pi * hour / 24).values,
    })
    dummies = pd.get_dummies(df["fmt"].values, prefix="fmt", dtype=float)
    return pd.concat([ef, dummies], axis=1)

def _raw_node_features(df, s, d, n):
    tmp = pd.DataFrame({"s": s, "d": d, "paid": df["amt_paid"].values, "recv": df["amt_recv"].values})
    out = tmp.groupby("s").agg(out_deg=("paid", "size"), out_amt=("paid", "mean"))
    inn = tmp.groupby("d").agg(in_deg=("recv", "size"), in_amt=("recv", "mean"))
    nf = pd.DataFrame(index=range(n)).join(out).join(inn).fillna(0)
    return np.log1p(nf)

def fit_stats(train_df, s, d, n):
    """Statistiques de normalisation calculées sur le TRAIN (sauvegardées dans la config)."""
    ef = _raw_edge_features(train_df)
    nf = _raw_node_features(train_df, s, d, n)
    return {
        "edge_cols": list(ef.columns),
        "amt_mean": {c: float(ef[c].mean()) for c in ["log_paid", "log_recv"]},
        "amt_std":  {c: float(ef[c].std())  for c in ["log_paid", "log_recv"]},
        "node_cols": list(nf.columns),
        "node_mean": [float(v) for v in nf.mean()],
        "node_std":  [float(v) + 1e-6 for v in nf.std()],
    }

def build_features(df, s, d, n, stats, node_rows=None):
    """Features d'arêtes (toutes les transactions) et de nœuds.
    node_rows : masque des transactions utilisées pour les agrégats de nœuds
    (le train à l'entraînement ; toutes les données fournies à l'inférence)."""
    ef = _raw_edge_features(df).reindex(columns=stats["edge_cols"], fill_value=0.0)
    for c in ["log_paid", "log_recv"]:
        ef[c] = (ef[c] - stats["amt_mean"][c]) / stats["amt_std"][c]
    if node_rows is None:
        node_rows = np.ones(len(df), dtype=bool)
    nf = _raw_node_features(df[node_rows], s[node_rows], d[node_rows], n)
    nf = (nf - np.array(stats["node_mean"])) / np.array(stats["node_std"])
    return (torch.tensor(ef.values, dtype=torch.float32),
            torch.tensor(nf.values, dtype=torch.float32))

def make_conv(kind, i, o):
    if kind == "gcn":  return GCNConv(i, o)
    if kind == "sage": return SAGEConv(i, o)
    if kind == "gat":  return GATConv(i, o // 4, heads=4)
    raise ValueError(kind)

class EdgeGNN(nn.Module):
    """GNN sur les nœuds (comptes) + MLP sur [h_src, h_dst, features de la transaction]."""
    def __init__(self, kind, in_node, in_edge, hid=64, layers=2, drop=0.2):
        super().__init__()
        dims = [in_node] + [hid] * layers
        self.convs = nn.ModuleList(make_conv(kind, dims[i], dims[i + 1]) for i in range(layers))
        self.drop = drop
        self.mlp = nn.Sequential(nn.Linear(2 * hid + in_edge, hid), nn.ReLU(),
                                 nn.Dropout(drop), nn.Linear(hid, 1))

    def forward(self, x, edge_index, label_index, edge_feat):
        for conv in self.convs:
            x = F.dropout(F.relu(conv(x, edge_index)), self.drop, self.training)
        h = torch.cat([x[label_index[0]], x[label_index[1]], edge_feat], dim=1)
        return self.mlp(h).squeeze(-1)

def best_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr * rc / (pr + rc + 1e-9)
    return float(th[np.argmax(f1[:-1])])

def compute_metrics(y, p, thr):
    y = np.asarray(y).astype(int); pred = (p >= thr).astype(int)
    tp = ((pred == 1) & (y == 1)).sum(); fp = ((pred == 1) & (y == 0)).sum(); fn = ((pred == 0) & (y == 1)).sum()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    k = max(int(y.sum()), 1); topk = np.argsort(-p)[:k]
    return dict(threshold=float(thr),
                pr_auc=float(average_precision_score(y, p)) if y.sum() > 0 else float("nan"),
                f1=float(2 * prec * rec / max(prec + rec, 1e-9)),
                precision=float(prec), recall=float(rec),
                recall_at_k=float(y[topk].mean()) if y.sum() > 0 else float("nan"))

## 4. Chargement des données et split temporel 60/20/20

In [ ]:
df_raw = pd.read_csv(CSV)
df = standardize_columns(df_raw)
del df_raw
df = df.sort_values("ts").reset_index(drop=True)

E = len(df)
i1, i2 = int(0.6 * E), int(0.8 * E)
df["split"] = np.where(np.arange(E) < i1, 0, np.where(np.arange(E) < i2, 1, 2))

src_np, dst_np, N, _ = index_accounts(df)
print(f"{N} comptes, {E} transactions")
print(df.groupby("split")["y"].agg(["count", "sum", "mean"]))

## 5. Features (statistiques du TRAIN uniquement)

In [ ]:
train_mask = (df["split"] == 0).values
STATS = fit_stats(df[train_mask], src_np[train_mask], dst_np[train_mask], N)

EF, X = build_features(df, src_np, dst_np, N, STATS, node_rows=train_mask)
Y  = torch.tensor(df["y"].values, dtype=torch.float32)
src, dst = torch.tensor(src_np), torch.tensor(dst_np)
sp = torch.tensor(df["split"].values)
print("Features arêtes :", EF.shape, "| features nœuds :", X.shape)
print("Colonnes arêtes :", STATS["edge_cols"])

## 6. Graphes et loaders (pas de fuite : le graphe de train ne voit que le train)

In [ ]:
def make_loader(graph_mask, target_mask, shuffle):
    ei = torch.stack([src[graph_mask], dst[graph_mask]])
    ei = torch.cat([ei, ei.flip(0)], dim=1)          # message passing dans les 2 sens
    data = Data(x=X, edge_index=ei, num_nodes=N)
    tgt = target_mask.nonzero().view(-1)
    return LinkNeighborLoader(
        data, num_neighbors=NUM_NEIGHBORS,
        edge_label_index=torch.stack([src[tgt], dst[tgt]]),
        edge_label=tgt,                               # id de la transaction
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2)

train_loader = make_loader(sp == 0, sp == 0, True)
val_loader   = make_loader(sp <= 1, sp == 1, False)
test_loader  = make_loader(sp <= 2, sp == 2, False)
EF_d, Y_d = EF.to(device), Y.to(device)

## 7. Entraînement et évaluation

In [ ]:
def run_batch(model, b):
    b = b.to(device); ids = b.edge_label
    return model(b.x, b.edge_index, b.edge_label_index, EF_d[ids]), Y_d[ids]

@torch.no_grad()
def predict(model, loader):
    model.eval(); ps, ys = [], []
    for b in loader:
        logit, y = run_batch(model, b)
        ps.append(torch.sigmoid(logit).cpu()); ys.append(y.cpu())
    return torch.cat(ps).numpy(), torch.cat(ys).numpy()

def train(kind, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = EdgeGNN(kind, X.shape[1], EF.shape[1], HID, LAYERS, DROP).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    pw = torch.tensor(POS_W, device=device)
    best_ap, best_state, history = -1, None, []
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for b in train_loader:
            opt.zero_grad()
            logit, y = run_batch(model, b)
            loss = F.binary_cross_entropy_with_logits(logit, y, pos_weight=pw)
            loss.backward(); opt.step(); tot += loss.item()
        p, y = predict(model, val_loader)
        ap = average_precision_score(y, p)
        history.append(ap)
        print(f"  [{kind}] epoch {ep+1:2d} | loss {tot/len(train_loader):.4f} | val PR-AUC {ap:.4f}")
        if ap > best_ap:
            best_ap = ap
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history

results, histories = [], {}
for kind in MODELS:
    for seed in SEEDS:
        print(f"=== {DATASET} | {kind} | seed {seed} ===")
        model, hist = train(kind, seed)
        histories[(kind, seed)] = hist

        pv, yv = predict(model, val_loader)
        thr = best_threshold(yv, pv)                  # seuil choisi sur la VALIDATION
        pt, yt = predict(model, test_loader)
        m = compute_metrics(yt, pt, thr)
        results.append(dict(dataset=DATASET, model=kind, seed=seed, **m))
        print("  TEST ->", {k: round(v, 4) for k, v in m.items()})

        tag = f"{kind}_{DATASET}_s{seed}"
        torch.save(model.state_dict(), os.path.join(OUT, f"{tag}.pt"))
        cfg = dict(conv=kind, dataset=DATASET, seed=seed, in_node=X.shape[1], in_edge=EF.shape[1],
                   hid=HID, layers=LAYERS, drop=DROP, num_neighbors=NUM_NEIGHBORS,
                   threshold=thr, stats=STATS, metrics=m)
        json.dump(cfg, open(os.path.join(OUT, f"{tag}_config.json"), "w"), indent=2)
        del model; torch.cuda.empty_cache()

## 8. Tableau récapitulatif et courbes

In [ ]:
import matplotlib.pyplot as plt

res = pd.DataFrame(results)
res.to_csv(os.path.join(OUT, f"benchmark_{DATASET}.csv"), index=False)
summary = res.groupby("model")[["pr_auc", "f1", "precision", "recall", "recall_at_k"]].agg(["mean", "std"])
display(summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for (kind, seed), h in histories.items():
    axes[0].plot(range(1, len(h) + 1), h, marker="o", label=f"{kind} (seed {seed})")
axes[0].set_title(f"PR-AUC validation — {DATASET}"); axes[0].set_xlabel("epoch"); axes[0].legend()
means = res.groupby("model")[["pr_auc", "f1"]].mean()
means.plot.bar(ax=axes[1], rot=0, title=f"Test — {DATASET}")
plt.tight_layout(); plt.show()

---
# 9. Inférence — tester les modèles sur vos propres données

Cette partie est **autonome** : après un redémarrage du runtime, exécutez seulement
les sections **1 (installation)**, **3 (fonctions communes)** puis celle-ci.

Choisissez la source des données dans `MODE` :

| `MODE` | Description |
|---|---|
| `"demo"` | prend les dernières transactions du CSV HI-Small (période de test, avec labels) |
| `"upload"` | vous téléversez un CSV depuis votre ordinateur |
| `"path"` | chemin d'un CSV déjà présent (Drive, etc.) |
| `"manual"` | quelques transactions saisies à la main dans la cellule |

**Format attendu** : le format IBM AML (`Timestamp, From Bank, Account, To Bank, Account.1,
Amount Received, Receiving Currency, Amount Paid, Payment Currency, Payment Format`),
la colonne `Is Laundering` étant **optionnelle** (si elle est présente, les métriques sont calculées).

⚠️ Un GNN juge une transaction d'après son **voisinage** : fournissez si possible un historique
de transactions autour de celles à analyser (paramètre `CONTEXT_CSV`), sinon un compte isolé
n'a presque pas de contexte. Les features de nœuds (degrés, montants moyens) sont recalculées
sur les données fournies : plus l'historique est court, plus elles s'éloignent de celles vues
à l'entraînement.

In [ ]:
import os, json, glob
from google.colab import drive
drive.mount('/content/drive')

DATASET = "HI-Small"
MODEL_DIR = f"/content/drive/MyDrive/delivery/{DATASET}"
MODELS_TO_USE = ["gcn", "sage", "gat"]
SEED = 0

MODE = "demo"           # "demo" | "upload" | "path" | "manual"
USER_CSV = "/content/drive/MyDrive/mes_transactions.csv"   # utilisé si MODE == "path"
DEMO_ROWS = 200_000                                         # utilisé si MODE == "demo"
CONTEXT_CSV = None      # optionnel : historique supplémentaire pour le contexte du graphe

print(sorted(os.path.basename(p) for p in glob.glob(os.path.join(MODEL_DIR, "*"))))

In [ ]:
# --- Chargement des données utilisateur ---
if MODE == "demo":
    full = pd.read_csv(f"/content/drive/MyDrive/{DATASET}_Trans.csv")
    full["_t"] = pd.to_datetime(full["Timestamp"])
    user_raw = full.sort_values("_t").tail(DEMO_ROWS).drop(columns="_t").reset_index(drop=True)  # période de test
    del full
elif MODE == "upload":
    from google.colab import files
    up = files.upload()
    user_raw = pd.read_csv(next(iter(up)))
elif MODE == "path":
    user_raw = pd.read_csv(USER_CSV)
elif MODE == "manual":
    user_raw = pd.DataFrame([
        # Timestamp, From Bank, Account, To Bank, Account.1, Amount Received, Receiving Currency,
        # Amount Paid, Payment Currency, Payment Format
        ["2022/09/10 10:00", 70, "ACC_A", 70, "ACC_B", 9500.0, "US Dollar", 9500.0, "US Dollar", "Cash"],
        ["2022/09/10 10:05", 70, "ACC_A", 12, "ACC_C", 9400.0, "US Dollar", 9400.0, "US Dollar", "Cash"],
        ["2022/09/10 10:10", 70, "ACC_A", 15, "ACC_D", 9300.0, "US Dollar", 9300.0, "US Dollar", "Cash"],
        ["2022/09/10 12:00", 12, "ACC_C", 99, "ACC_E", 9000.0, "US Dollar", 9000.0, "US Dollar", "ACH"],
        ["2022/09/10 12:30", 15, "ACC_D", 99, "ACC_E", 8900.0, "US Dollar", 8900.0, "US Dollar", "ACH"],
        ["2022/09/10 13:00", 99, "ACC_E", 70, "ACC_A", 17500.0, "US Dollar", 17500.0, "US Dollar", "Wire"],
    ], columns=IBM_COLUMNS[:10])
else:
    raise ValueError(MODE)

user = standardize_columns(user_raw)
user["_target"] = True
if CONTEXT_CSV:
    ctx = standardize_columns(pd.read_csv(CONTEXT_CSV)); ctx["_target"] = False
    user = pd.concat([ctx, user], ignore_index=True)
user = user.sort_values("ts").reset_index(drop=True)
has_labels = user["y"].notna().all()
print(f"{int(user['_target'].sum())} transactions à scorer, {len(user)} au total dans le graphe, labels : {has_labels}")
display(user.head())

In [ ]:
# --- Chargement des modèles et scoring ---
def load_model(kind):
    tag = f"{kind}_{DATASET}_s{SEED}"
    cfg = json.load(open(os.path.join(MODEL_DIR, f"{tag}_config.json")))
    m = EdgeGNN(cfg["conv"], cfg["in_node"], cfg["in_edge"], cfg["hid"], cfg["layers"], cfg["drop"])
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f"{tag}.pt"), map_location="cpu"))
    return m.to(device).eval(), cfg

@torch.no_grad()
def score(model, cfg, data_df):
    s_np, d_np, n, _ = index_accounts(data_df)
    ef, x = build_features(data_df, s_np, d_np, n, cfg["stats"])
    s, d = torch.tensor(s_np), torch.tensor(d_np)
    ei = torch.cat([torch.stack([s, d]), torch.stack([d, s])], dim=1)
    tgt = torch.tensor(np.where(data_df["_target"].values)[0])
    loader = LinkNeighborLoader(
        Data(x=x, edge_index=ei, num_nodes=n), num_neighbors=cfg["num_neighbors"],
        edge_label_index=torch.stack([s[tgt], d[tgt]]), edge_label=tgt,
        batch_size=4096, shuffle=False)
    ef_d = ef.to(device); out = np.zeros(len(data_df), dtype=np.float32)
    for b in loader:
        b = b.to(device); ids = b.edge_label
        p = torch.sigmoid(model(b.x, b.edge_index, b.edge_label_index, ef_d[ids]))
        out[ids.cpu().numpy()] = p.cpu().numpy()
    return out[data_df["_target"].values]

target = user[user["_target"]].copy()
for kind in MODELS_TO_USE:
    model, cfg = load_model(kind)
    target[f"score_{kind}"] = score(model, cfg, user)
    target[f"alert_{kind}"] = (target[f"score_{kind}"] >= cfg["threshold"]).astype(int)
    if has_labels:
        m = compute_metrics(target["y"].values, target[f"score_{kind}"].values, cfg["threshold"])
        print(f"[{kind}] seuil={cfg['threshold']:.3f} |", {k: round(v, 4) for k, v in m.items() if k != "threshold"})
    else:
        print(f"[{kind}] seuil={cfg['threshold']:.3f} | {target[f'alert_{kind}'].sum()} alertes")

score_cols = [f"score_{k}" for k in MODELS_TO_USE]
target["score_moyen"] = target[score_cols].mean(axis=1)
target["nb_modeles_en_alerte"] = target[[f"alert_{k}" for k in MODELS_TO_USE]].sum(axis=1)

In [ ]:
# --- Transactions les plus suspectes ---
cols = ["ts", "src_key", "dst_key", "amt_paid", "cur_paid", "fmt"] + score_cols + ["score_moyen", "nb_modeles_en_alerte"]
if has_labels: cols.append("y")
TOP = 20
display(target.sort_values("score_moyen", ascending=False)[cols].head(TOP))

out_path = os.path.join(MODEL_DIR, f"predictions_{MODE}.csv")
target[cols].to_csv(out_path, index=False)
print("Prédictions sauvegardées :", out_path)